# ATML PA3 · Task 3 — live generation (Colab T4)

Only Step 4 (parallel vs sequential vs single long response, 50 MATH questions, Qwen2.5-3B-Instruct,
batch size 1) needs a GPU; the offline Steps 1-3 and 5-6 are CPU-only and their results are already
in `results/task3/`. Every finished question is appended to `results/task3/live/generations.jsonl`
and pushed to the HF store, so if Colab disconnects just run all cells again.

Expected time on a T4: roughly 2-5 h for 50 questions (≤ 8,192 new tokens per question).

**Secrets** (Kaggle: *Add-ons → Secrets*; Colab: key icon): `HF_TOKEN` (write access; used for the
private checkpoint store and model downloads) and `GITHUB_TOKEN` (read access to the private repo;
not needed once the repo is public).

In [ ]:
# --- Run configuration ---
REPO = "https://github.com/rana-taqveem/atml_internal.git"
BRANCH = "main"
STORE_REPO = ""          # "" -> <your-hf-username>/atml-pa3-store (private, created automatically)
LIMIT = 0                # 0 = all 50 questions; e.g. 2 for a quick smoke test
RUN_OFFLINE = False      # True re-runs the CPU-only offline analysis (~30-60 min)

In [ ]:
# --- Setup: secrets, clone, install (identical in every task notebook) ---
import os, subprocess, sys

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient          # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata                       # Colab
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN")
if STORE_REPO:
    os.environ["PA3_STORE_REPO"] = STORE_REPO
gh = secret("GITHUB_TOKEN")
url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
# Kaggle: /kaggle/working is capped at 20 GB and saved with the notebook, so the repo and all
# checkpoints live in /tmp; only result zips are written to /kaggle/working.
KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else "/content"
CLONE = "/tmp/pa3" if KAGGLE else "/content/pa3"
if not os.path.exists(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CLONE], check=True)
else:
    subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=True)
os.chdir(f"{CLONE}/assignment_03/ATML-PA3")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Data: supplied rollouts (Google Drive) + fixed splits ---
!python -m task3_ttc.data
if RUN_OFFLINE:
    !python -m task3_ttc.grading
    !python -m task3_ttc.offline
    !python -m task3_ttc.signals
    !python -m task3_ttc.adaptive

In [ ]:
# --- Step 4: live generation (resumable) ---
!python -m task3_ttc.live {"--limit " + str(LIMIT) if LIMIT else ""}

In [ ]:
!python -m task3_ttc.summarize
ZIP_NAME, RESULTS_DIR = 'task3_results', 'results/task3'
# --- Collect: small results into a zip next to the notebook output (also mirrored in the HF store) ---
import shutil
shutil.make_archive(f"{WORK}/{ZIP_NAME}", "zip", ".", RESULTS_DIR)
if os.path.exists("logs"):
    shutil.make_archive(f"{WORK}/{ZIP_NAME}_logs", "zip", ".", "logs")
print("wrote", f"{WORK}/{ZIP_NAME}.zip")